In [1]:
#Word tokenization

In [2]:
import re

text = "Hello world! I love Python."
tokens = re.split(r"\W+", text)  # split by non-word characters
tokens = [t for t in tokens if t]  # remove empty tokens
print(tokens)
# ['Hello', 'world', 'I', 'love', 'Python']


['Hello', 'world', 'I', 'love', 'Python']


In [3]:
#NLTK

In [4]:
import nltk
# Tokenizers
nltk.download('punkt')
nltk.download('punkt_tab')

# Stopwords
nltk.download('stopwords')

# Lemmatizer
nltk.download('wordnet')

# POS tagging
nltk.download('averaged_perceptron_tagger')


from nltk.tokenize import word_tokenize

text = "Hello world! I love Python."
tokens = word_tokenize(text)
print(tokens)
# ['Hello', 'world', '!', 'I', 'love', 'Python', '.']


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\KRISHNENDU\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\KRISHNENDU\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\KRISHNENDU\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\KRISHNENDU\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\KRISHNENDU\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


['Hello', 'world', '!', 'I', 'love', 'Python', '.']


What is Byte Pair Encoding?

Start with characters as tokens.
Repeatedly merge the most frequent pair of symbols (characters or subwords) into a single token.
Continue until you reach the desired vocabulary size.


word = "lower"
Initial tokens: l o w e r
If "lo" is frequent → merge → lo w e r
If "er" is frequent → merge → lo w er


In [5]:
from collections import Counter

def get_stats(vocab):
    """Count frequency of symbol pairs in vocab"""
    pairs = Counter()
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols)-1):
            pairs[(symbols[i], symbols[i+1])] += freq
    return pairs

def merge_vocab(pair, vocab):
    """Merge the most frequent pair in vocab"""
    bigram = ' '.join(pair)
    new_vocab = {}
    for word in vocab:
        new_word = word.replace(bigram, ''.join(pair))
        new_vocab[new_word] = vocab[word]
    return new_vocab

# Example training data (word frequencies)
vocab = {
    "l o w": 5,
    "l o w e r": 2,
    "n e w e r": 6,
    "w i d e r": 3
}

print("Initial Vocab:", vocab)

num_merges = 10
for i in range(num_merges):
    pairs = get_stats(vocab)
    if not pairs:
        break
    best = max(pairs, key=pairs.get)
    vocab = merge_vocab(best, vocab)
    print(f"Step {i+1}: Merge {best}")
    print("Updated Vocab:", vocab)


Initial Vocab: {'l o w': 5, 'l o w e r': 2, 'n e w e r': 6, 'w i d e r': 3}
Step 1: Merge ('e', 'r')
Updated Vocab: {'l o w': 5, 'l o w er': 2, 'n e w er': 6, 'w i d er': 3}
Step 2: Merge ('w', 'er')
Updated Vocab: {'l o w': 5, 'l o wer': 2, 'n e wer': 6, 'w i d er': 3}
Step 3: Merge ('l', 'o')
Updated Vocab: {'lo w': 5, 'lo wer': 2, 'n e wer': 6, 'w i d er': 3}
Step 4: Merge ('n', 'e')
Updated Vocab: {'lo w': 5, 'lo wer': 2, 'ne wer': 6, 'w i d er': 3}
Step 5: Merge ('ne', 'wer')
Updated Vocab: {'lo w': 5, 'lo wer': 2, 'newer': 6, 'w i d er': 3}
Step 6: Merge ('lo', 'w')
Updated Vocab: {'low': 5, 'lower': 2, 'newer': 6, 'w i d er': 3}
Step 7: Merge ('w', 'i')
Updated Vocab: {'low': 5, 'lower': 2, 'newer': 6, 'wi d er': 3}
Step 8: Merge ('wi', 'd')
Updated Vocab: {'low': 5, 'lower': 2, 'newer': 6, 'wid er': 3}
Step 9: Merge ('wid', 'er')
Updated Vocab: {'low': 5, 'lower': 2, 'newer': 6, 'wider': 3}


Example with SentencePiece (subword tokenizer library)

In [6]:
import sentencepiece as spm

# Train with smaller vocab size
spm.SentencePieceTrainer.train(
    input="my_corpus.txt",
    model_prefix="subword",
    vocab_size=50,          # keep it small
    character_coverage=1.0, # ensure all characters are covered
    model_type="bpe"        # or "unigram" (default)
)

# Load tokenizer
sp = spm.SentencePieceProcessor(model_file="subword.model")

# Encode + decode
text = "unhappiness is increasing rapidly"
print("Encoded:", sp.encode(text, out_type=str))
print("Decoded:", sp.decode(sp.encode(text)))


Encoded: ['▁', 'un', 'h', 'appiness', '▁is', '▁', 'in', 'cr', 'e', 'as', 'ing', '▁', 'r', 'ap', 'i', 'dl', 'y']
Decoded: unhappiness is increasing rapidly


In [7]:
#Hugging Face Tokenizers

In [8]:
from tokenizers import ByteLevelBPETokenizer

tokenizer = ByteLevelBPETokenizer()
tokenizer.train(files="my_corpus.txt", vocab_size=2000, min_frequency=2)

print(tokenizer.encode("unhappiness is increasing rapidly").tokens)


['un', 'happiness', 'Ġis', 'Ġ', 'in', 'c', 'r', 'e', 'a', 's', 'ing', 'Ġ', 'r', 'ap', 'i', 'd', 'l', 'y']


What is WordPiece?

Similar to BPE, but instead of just merging the most frequent pair, it uses a likelihood-based criterion.
The goal is to maximize the probability of the training data under a language model.
Result: Better handling of rare words and languages with complex morphology.

🔹 How WordPiece Works

Start with characters as the initial vocabulary.
Example: u n h a p p i n e s s

At each step:

Instead of merging the most frequent pair,
WordPiece picks the pair that maximizes the likelihood of the corpus.
Keep merging until the desired vocab size is reached.

In [ ]:
from transformers import BertTokenizer

# Load pretrained WordPiece tokenizer (from BERT base)
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

text = "unhappiness is increasing rapidly"
tokens = tokenizer.tokenize(text)
print(tokens)


C:\Users\KRISHNENDU\anaconda3\envs\etl-pipeline\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
